In [2]:
!pip -q install pytorch-ignite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.8/371.8 kB 8.3 MB/s eta 0:00:00


In [3]:
import torch
import pandas as pd

from sklearn.model_selection import train_test_split
from ignite.metrics import MaximumMeanDiscrepancy
from google.colab import files


# --------------------------------------------------
# 1. UPLOAD THE BITSTRING DATASET
# --------------------------------------------------

print("Upload bitstrings.csv")

uploaded = files.upload()

# Automatically use the file you upload
csv_file = next(iter(uploaded.keys()))

print("Using file:", csv_file)


# --------------------------------------------------
# 2. LOAD AND PREPARE THE DATA
# --------------------------------------------------

bitstring_df = pd.read_csv(
    csv_file,
    dtype={"bitstring": str}
)

# Convert each bitstring:
# "010101" -> [0, 1, 0, 1, 0, 1]

data = []

for bitstring in bitstring_df["bitstring"]:

    bitstring = bitstring.strip()

    sample = [
        int(bit)
        for bit in bitstring
    ]

    data.append(sample)


data = torch.tensor(
    data,
    dtype=torch.float32
)


print("\nDataset shape:", data.shape)
print("Number of samples:", data.shape[0])
print("Bits per sample:", data.shape[1])

print("\nFirst sample:")
print(data[0])


# --------------------------------------------------
# 3. SPLIT DATA - 80% TRAINING / 20% TESTING
# --------------------------------------------------

# Convert to NumPy temporarily because
# sklearn's train_test_split works cleanly with it

train_np, test_np = train_test_split(
    data.numpy(),
    train_size=0.80,
    random_state=50,
    shuffle=True
)


# Convert back to PyTorch tensors

train_data = torch.tensor(
    train_np,
    dtype=torch.float32
)

test_data = torch.tensor(
    test_np,
    dtype=torch.float32
)


print("\nTraining data:", train_data.shape)
print("Testing data:", test_data.shape)


# --------------------------------------------------
# 4. BAYESIAN INFERENCE MODEL
# --------------------------------------------------

# Beta(1,1) prior
# This means we initially treat 0 and 1
# as equally likely.

alpha_prior = 1.0
beta_prior = 1.0


# Count how many 1s occur at every bit position

ones = train_data.sum(dim=0)


# Count how many 0s occur at every bit position

zeros = train_data.shape[0] - ones


# Update the Bayesian posterior

alpha_posterior = alpha_prior + ones
beta_posterior = beta_prior + zeros


# Expected probability of seeing a 1
# at each position

bit_probabilities = (
    alpha_posterior
    /
    (alpha_posterior + beta_posterior)
)


print("\nFirst 10 learned probabilities:")

print(
    bit_probabilities[:10]
)


# --------------------------------------------------
# 5. GENERATE NEW BITSTRING SAMPLES
# --------------------------------------------------

# Use a fixed seed so results are reproducible

torch.manual_seed(50)


# Generate the same number of samples
# as we have in the testing dataset

num_samples = test_data.shape[0]


probability_matrix = (
    bit_probabilities
    .unsqueeze(0)
    .repeat(num_samples, 1)
)


generated_samples = torch.bernoulli(
    probability_matrix
)


print(
    "\nGenerated samples:",
    generated_samples.shape
)

print("\nFirst generated sample:")

print(
    generated_samples[0]
)


# --------------------------------------------------
# 6. MMD EVALUATION
# --------------------------------------------------

# Compare the model-generated distribution
# against the real testing data.

mmd = MaximumMeanDiscrepancy()


mmd.update(
    (
        generated_samples.float(),
        test_data.float()
    )
)


score = mmd.compute()


print("\n-------------------------")
print("MMD RESULTS")
print("-------------------------")

print(
    "MMD:",
    float(score)
)


# --------------------------------------------------
# 7. MODEL SUMMARY
# --------------------------------------------------

print("\n-------------------------")
print("MODEL SUMMARY")
print("-------------------------")

print(
    "Training samples:",
    train_data.shape[0]
)

print(
    "Testing samples:",
    test_data.shape[0]
)

print(
    "Bitstring length:",
    train_data.shape[1]
)

print(
    "Parameters stored by Bayesian model:",
    len(bit_probabilities)
)

print(
    "Final MMD score:",
    float(score)
)

Upload bitstrings.csv


Saving bitstrings.csv to bitstrings.csv
Using file: bitstrings.csv

Dataset shape: torch.Size([1000, 128])
Number of samples: 1000
Bits per sample: 128

First sample:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])

Training data: torch.Size([800, 128])
Testing data: torch.Size([200, 128])

First 10 learned probabilities:
tensor([0.5087, 0.5150, 0.5137, 0.5087, 0.5137, 0.5137, 0.5187, 0.5200, 0.5224,
        0.5262])

Generated samples: torch.Size([200, 128])

First 